# Coat64 Fast SWA submission

Export and validate the locally generated prediction CSV for the notebook-only competition submission flow.


In [ ]:
from pathlib import Path
import pandas as pd

input_root = Path("/kaggle/input")
print("Input mounts:", [p.name for p in input_root.iterdir()])
candidates = []
for mount in input_root.iterdir():
    if not mount.is_dir() or mount.name == "competitions":
        continue
    # Dataset mounts may add an owner directory, so inspect at most three levels.
    for p in [mount / "submission.csv", *mount.glob("*/submission.csv"), *mount.glob("*/*/submission.csv")]:
        if p.is_file():
            candidates.append(p)
if len(candidates) != 1:
    raise RuntimeError(f"Expected one prediction CSV, found {candidates}")
source = candidates[0]
pred = pd.read_csv(source, dtype={"StudyInstanceUID": str})
if pred["StudyInstanceUID"].duplicated().any():
    raise ValueError("Duplicate StudyInstanceUID rows")
if not pred.iloc[:, 1:].notna().all().all():
    raise ValueError("Prediction values contain NaN")
output = Path("/kaggle/working/submission.csv")
pred.to_csv(output, index=False)
print(f"Copied {source} to {output}: {len(pred)} studies x {len(pred.columns)} columns")
print(pred.head().to_string(index=False))
